# ДЗ1. ReAct-агент підтримки гравців у LangGraph

Агент відповідає на звернення гравців мобільної гри «Forge & Craft»: покупки, краші, сервери, компенсації.
Код лежить у репозиторії; ноутбук клонує його, запускає юніт-тести, тест-кейси й демонстрації захисних механізмів.

**Проєктні рішення**
- Домен — підтримка гравців мобільної гри: шість інструментів на тестових даних.
- Модель — Claude Sonnet 5.5 (`claude-sonnet-5-5`) через LangChain (`ChatAnthropic`).
- Фінальна відповідь — структурований `SupportAnswer`: відповідь, категорія, чи вирішено, наступна дія, докази з інструментів, компенсація.
- Ліміти: 10 викликів моделі, 120 с на звернення, зациклення — 3 однакові виклики інструмента поспіль.
- Тестування: 5 тест-кейсів різної складності + 3 окремі демонстрації лімітів.

In [ ]:
# Пробний прогін: один тест-кейс без демо. Для повного прогону — False.
DRY_RUN = True

REPO_URL = "https://github.com/<логін>/hw1-react-agent.git"
REPO_DIR = "hw1-react-agent"

In [ ]:
import os
import sys

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
!pip install -q -r requirements.txt

In [ ]:
# Ключ береться з Colab Secrets і не зберігається в ноутбуці.
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Завдання 1. Інструменти з Pydantic-схемами

Кожен інструмент має закриту схему (`extra="forbid"`), `Field(description=...)` для моделі та `field_validator`,
який нормалізує ввід (`p1001 → P-1001`, `v2.3.1 → 2.3.1`, «Європа» → `eu`) або відхиляє його зрозумілим повідомленням.
Юніт-тести перевіряють інструменти, захисні механізми й граф на сценарній моделі — без викликів API.

In [ ]:
!python -m pytest -v

In [ ]:
# Як агент бачить помилку валідації: замість винятку — текст, за яким модель виправляє аргументи.
from pydantic import ValidationError

from agent import format_validation_error
from tools import ClientCheckInput, PlayerProfileInput, check_client

for schema, tool_name, args in [
    (ClientCheckInput, "check_client", {"platform": "android", "app_version": "latest"}),
    (PlayerProfileInput, "get_player_profile", {"player_id": "1001", "lang": "uk"}),
]:
    try:
        schema.model_validate(args)
    except ValidationError as exc:
        print(format_validation_error(tool_name, exc))

print(check_client.invoke({"platform": "iPhone", "app_version": "v2.0.4"}))

## Завдання 2. LangGraph з циклом LLM → tools → LLM

Граф: `agent` (модель з інструментами) → умовне ребро → `tools` → назад до `agent`; коли викликів інструментів немає
або спрацював ліміт — `finalize`, який повертає `SupportAnswer`.

In [ ]:
from IPython.display import Image, display

from agent import SupportAgent, make_model

model = make_model()
agent = SupportAgent(model)

try:
    display(Image(agent.graph.get_graph().draw_mermaid_png()))
except Exception:
    print(agent.graph.get_graph().draw_mermaid())

In [ ]:
import json

result = agent.run("Привіт! Чи працюють зараз сервери в Європі?", run_id="demo-single")
print("Зупинка:", result["stop_reason"], "| кроків:", result["steps"], "| інструменти:", result["tool_calls"])
print(json.dumps(result["final"], ensure_ascii=False, indent=2))

## Завдання 3–4. Тест-кейси, демонстрації лімітів, логування

`test_runner.main()` проганяє тест-кейси й демо, пише `test_results.json`, `demo_results.json` і `trajectory.json`.
Статус тест-кейсу: `success` — усі перевірки пройдено; `partial` — відповідь є, але частина перевірок не пройшла;
`fail` — агент зупинився за лімітом.

In [ ]:
import test_runner

results, demos = test_runner.main(
    model=model,
    only=["TC-001"] if DRY_RUN else None,
    skip_demos=DRY_RUN,
)

In [ ]:
import pandas as pd

columns = ["test_id", "complexity", "steps", "tool_calls_count", "time_ms", "status", "stop_reason"]
display(pd.DataFrame(results)[columns])
if demos:
    display(pd.DataFrame(demos)[["demo_id", "expected_stop", "actual_stop", "triggered", "steps", "time_ms"]])
print(json.dumps(json.load(open("test_results.json", encoding="utf-8"))["summary"], ensure_ascii=False, indent=2))

In [ ]:
# Перевірки кожного тест-кейсу: що саме пройшло, а що ні.
for r in results:
    failed = [name for name, ok in r["checks"].items() if not ok]
    print(r["test_id"], r["status"], "— не пройшли:", failed or "—")

In [ ]:
from google.colab import files

for name in ["test_results.json", "demo_results.json", "trajectory.json"]:
    if os.path.exists(name):
        files.download(name)